In [2]:
import json
import re
import ast

pattern = r"\[.*\]"
with open("F1_score_calculator.ipynb") as file:
    jsonContent = json.load(file)
phi4List = []
for i in jsonContent['cells']:
    a = [j for j in i['source'] if "chat" in j][0]
    b = ast.literal_eval(re.search(pattern, a)[0])
    phi4List.append(b)
print(phi4List)

[['challenging', 'uncooperative', 'noncompliant', 'cursing at nurses', 'difficult patient', 'challenging interactions', 'refused several exams', 'resistant to modifying his diet', 'noncompliance'], ['uncooperative', 'noncompliance', 'refusing', 'complicating', 'compliance issues'], ['challenging', 'difficult', 'noncompliant', 'refusing', 'contradicting themselves', 'resistant', 'inconsistent in narration', 'exhibited compliance issues', 'declining exams', 'noncompliant behavior'], ['non-compliant', 'frequently missed', 'medication adherence issues', 'frequently consuming alcohol over the recommended guidelines', 'refused examination and medication adjustments', 'difficult and challenging patient', 'inconsistent historical reporting'], ['difficult attitude', 'resistant', 'challenging', 'nonadherence', 'medication nonadherence', 'significant concern', 'declining to participate', 'noncompliant'], ['difficult', 'noncompliant', 'frequent visitor to ED', 'nonadherence', 'refused', 'inconsist

In [8]:
import pandas as pd
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="phi4")

def askOllama(prompt):
    result = model.invoke(input=prompt)
    return result

df = pd.read_csv("/Users/sagewong/git/StigmatizingLanguageProject/FinalFinalAnnnotatedData.csv")
allList = []
for index, i in enumerate(phi4List):
    clinicalNote = df.iloc[index]["Completion"]
    clinicalNote = re.sub(r'^.*?\*\*History of Present Illness:\*\*', '', clinicalNote, flags=re.DOTALL)
    sentences = clinicalNote.split("**")
    sentences = [item for part in sentences for item in part.split("-")]
    for word in i:
        if len([j for j in sentences if word in j]) > 0:
            text = [j for j in sentences if word in j][0]
            allList.append([word, text])

def group_by_second_index(data):
    result = {}

    for element in data:
        key = element[1]  # The second index (the grouping key)
        value = element[0]  # The first index (the value for the key)

        if key in result:
            result[key].append(value)  # If the key exists, append the value to the list
        else:
            result[key] = [value]  # If the key doesn't exist, create a new list with the value

    return result

z = group_by_second_index(allList)
z

{' Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.\n\n': ['challenging',
  'uncooperative',
  'noncompliant',
  'cursing at nurses'],
 ' Throughout his stay, Mohammad was a difficult patient, reflecting a pattern of challenging interactions and unreliable historical recounting. Compliance issues were evident, as he refused several exams and was resistant to modifying his diet, despite clear evidence linking his nutritional habits to his short stature. Cursing at staff persisted, complicating our attempts to provide care.\n\n': ['difficult patient',
  'challenging interactions',
  'refused several exams',
  'resistant to mo

In [10]:
print(len(z.items()))

166
